# Notebook 3 — ECGRiskNet-XAI Model Architecture

## Full Architecture (12-Lead Only)
```
Input ECG  (B, 12, 1000)
    ↓
Conv Stem              → (B, 64, 500)
    ↓
InceptionTime Block    → (B, 128, 500)  [kernels 3,7,15,31 capture P,QRS,ST,T]
    ↓
Residual Block ×2      → (B, 256, 250)
    ↓
SE Attention           → (B, 256, 250)  [lead-wise importance]
    ↓
Transformer Encoder ×4 → (B, 125, 256)  [long-range dependencies]
    ↓
Attention Pooling      → (B, 256)       [explainable pooling]
    ↓
Metadata Fusion        → (B, 256+16)    [age, sex, heart rate]
    ↓
Shared Embedding       → (B, 256)
    ↓
SupCon Projection      → (B, 128)       [contrastive learning head]
    ↓
Multi-Task Heads:
    Risk Head          → (B, 4)         [Low/Moderate/High/Critical]
    Arrhythmia Head    → (B, 6)         [rhythm classes]
    MI Head            → (B, 2)         [MI / No MI]
    ST_T Head          → (B, 3)         [ST-T change classes]
    Conduction Head    → (B, 4)         [conduction defect classes]
    ↓
CB-Focal Loss + SupCon Loss
    ↓
4-Tier Explainable Output
```

## Key Architecture Changes from Previous Version
1. ✅ **InceptionTime** replaces simple CNN stem — captures P-wave, QRS, ST, T-wave simultaneously
2. ✅ **SE Attention** (Squeeze-Excite) provides lead-wise importance for SHAP
3. ✅ **4× Transformer Encoder** replaces Dilated TCN — better long-range dependencies
4. ✅ **Attention Pooling** replaces AdaptiveAvgPool — produces explainable per-timestep weights
5. ✅ **Metadata MLP Fusion** (age, sex, HR) — feeds Tier 3 clinical interpretation
6. ✅ **SupCon Projection Head** — for Supervised Contrastive Loss during training
7. ✅ **5 Multi-Task Heads** — Risk, Arrhythmia, MI, ST/T, Conduction
8. ✅ **CB-Focal Loss** (Class-Balanced) replaces plain Focal Loss
9. ✅ All norms: GroupNorm / LayerNorm (stable for all batch sizes, no BatchNorm)


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import json
from pathlib import Path
from typing import Optional, Dict, Tuple

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device   = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SAVE_DIR = Path("./ECG_Project/data")

with open(SAVE_DIR / "metadata.json") as f:
    META = json.load(f)
RISK_LABELS = {int(k): v for k, v in META["risk_labels"].items()}
NUM_CLASSES = 4
META_DIM    = 3   # age_norm, sex, hr_norm
print(f"Device: {device}")


## Building Blocks

In [ ]:
# ─── 1. Residual Block ────────────────────────────────────────────────────────
class ResBlock(nn.Module):
    """Residual CNN block with GroupNorm."""
    def __init__(self, ch, k=7, dropout=0.2):
        super().__init__()
        p = k // 2
        self.net = nn.Sequential(
            nn.Conv1d(ch, ch, k, padding=p, bias=False),
            nn.GroupNorm(min(8, ch), ch), nn.GELU(), nn.Dropout(dropout),
            nn.Conv1d(ch, ch, k, padding=p, bias=False),
            nn.GroupNorm(min(8, ch), ch), nn.GELU(),
        )
    def forward(self, x): return x + self.net(x)


# ─── 2. InceptionTime Block ───────────────────────────────────────────────────
class InceptionBlock(nn.Module):
    """
    InceptionTime-style block with 4 parallel kernel sizes.
    Kernels [3,7,15,31] capture P-wave, QRS complex, ST segment, T-wave respectively.
    """
    def __init__(self, in_ch, out_ch_per_branch=32, bottleneck=32, dropout=0.1):
        super().__init__()
        # Bottleneck reduces channels before parallel convolutions
        self.bottleneck = nn.Conv1d(in_ch, bottleneck, 1, bias=False)

        self.conv3  = nn.Conv1d(bottleneck, out_ch_per_branch, 3,  padding=1,  bias=False)
        self.conv7  = nn.Conv1d(bottleneck, out_ch_per_branch, 7,  padding=3,  bias=False)
        self.conv15 = nn.Conv1d(bottleneck, out_ch_per_branch, 15, padding=7,  bias=False)
        self.conv31 = nn.Conv1d(bottleneck, out_ch_per_branch, 31, padding=15, bias=False)

        # MaxPool branch for residual detail
        self.maxpool_branch = nn.Sequential(
            nn.MaxPool1d(3, stride=1, padding=1),
            nn.Conv1d(in_ch, out_ch_per_branch, 1, bias=False),
        )

        total_out = out_ch_per_branch * 5
        self.norm = nn.GroupNorm(min(8, total_out), total_out)
        self.act  = nn.GELU()
        self.drop = nn.Dropout(dropout)

        # Projection to merge branches and match out_ch
        self.proj = nn.Conv1d(total_out, out_ch_per_branch * 4, 1, bias=False)

    def forward(self, x):
        b = self.bottleneck(x)
        branches = [
            self.conv3(b),
            self.conv7(b),
            self.conv15(b),
            self.conv31(b),
            self.maxpool_branch(x),
        ]
        out = torch.cat(branches, dim=1)
        out = self.drop(self.act(self.norm(out)))
        return self.proj(out)   # (B, 4*out_ch_per_branch, T)


# ─── 3. Squeeze-Excite (SE) Attention ────────────────────────────────────────
class SEBlock(nn.Module):
    """
    Channel (lead-wise) Squeeze-Excite attention.
    Provides per-lead importance weights captured for SHAP / lead importance plot.
    """
    def __init__(self, ch, reduction=16):
        super().__init__()
        mid = max(ch // reduction, 4)
        self.fc = nn.Sequential(
            nn.Linear(ch, mid), nn.ReLU(),
            nn.Linear(mid, ch), nn.Sigmoid(),
        )
        self.last_weights: Optional[torch.Tensor] = None

    def forward(self, x):
        # x: (B, C, T)
        w = self.fc(x.mean(dim=-1))  # (B, C)
        self.last_weights = w.detach().cpu()
        return x * w.unsqueeze(-1)


# ─── 4. Transformer Encoder Block ────────────────────────────────────────────
class TransformerBlock(nn.Module):
    """
    Standard Transformer block: Multi-Head Self-Attention + FFN.
    Stores attention weights for Tier 2 multi-head attention maps.
    """
    def __init__(self, d_model, heads=8, dropout=0.1, ffn_mult=4):
        super().__init__()
        self.attn  = nn.MultiheadAttention(d_model, heads, dropout=dropout, batch_first=True)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn   = nn.Sequential(
            nn.Linear(d_model, d_model * ffn_mult),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * ffn_mult, d_model),
            nn.Dropout(dropout),
        )
        self.last_attn_weights: Optional[torch.Tensor] = None

    def forward(self, x):
        # x: (B, T, d_model)
        out, w = self.attn(x, x, x, need_weights=True, average_attn_weights=False)
        if w is not None:
            self.last_attn_weights = w.detach().cpu()  # (B, heads, T, T)
        x = self.norm1(x + out)
        x = self.norm2(x + self.ffn(x))
        return x


# ─── 5. Attention Pooling ─────────────────────────────────────────────────────
class AttentionPooling(nn.Module):
    """
    Learnable attention-weighted pooling over time dimension.
    Provides per-timestep importance for Tier 2 heatmap visualisation.
    """
    def __init__(self, d_model):
        super().__init__()
        self.query = nn.Linear(d_model, 1)
        self.last_pool_weights: Optional[torch.Tensor] = None

    def forward(self, x):
        # x: (B, T, d_model)
        w = torch.softmax(self.query(x), dim=1)  # (B, T, 1)
        self.last_pool_weights = w.detach().cpu()
        return (x * w).sum(dim=1)                # (B, d_model)


# ─── 6. Metadata MLP ─────────────────────────────────────────────────────────
class MetadataMLP(nn.Module):
    """Encodes (age_norm, sex, hr_norm) into a fixed-dim embedding."""
    def __init__(self, in_dim=3, out_dim=16):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 32), nn.GELU(),
            nn.Linear(32, out_dim), nn.GELU(),
        )
    def forward(self, m): return self.net(m)  # (B, out_dim)


print("✅ All building blocks defined.")


## Full Model: ECGRiskNetXAI

In [ ]:
class ECGRiskNetXAI(nn.Module):
    """
    ECGRiskNet-XAI: Full 4-Tier Explainable AI Architecture

    Input:
        ecg  : (B, 12, 1000)
        meta : (B, 3)   [age_norm, sex, hr_norm]

    Output dict:
        risk        : (B, 4)   — Tier 1 main risk head
        arrhythmia  : (B, 6)   — Tier 3 rhythm head
        mi          : (B, 2)   — Tier 3 MI head
        st_t        : (B, 3)   — Tier 3 ST/T change head
        conduction  : (B, 4)   — Tier 3 conduction defect head
        projection  : (B, 128) — SupCon embedding (used in loss, not final output)
    """
    def __init__(self, in_ch=12, base_ch=64, num_risk_classes=4,
                 meta_dim=3, dropout=0.3):
        super().__init__()

        # ── Stage 1: Conv Stem ────────────────────────────────────────────────
        self.stem = nn.Sequential(
            nn.Conv1d(in_ch, base_ch, kernel_size=15, padding=7, bias=False),
            nn.GroupNorm(min(8, base_ch), base_ch),
            nn.GELU(),
            nn.MaxPool1d(2),   # 1000 → 500
        )

        # ── Stage 2: InceptionTime Block ─────────────────────────────────────
        self.inception = InceptionBlock(
            in_ch=base_ch,
            out_ch_per_branch=32,
            bottleneck=base_ch,
            dropout=dropout * 0.5,
        )   # out: (B, 128, 500)
        incep_ch = 32 * 4  # 128

        # ── Stage 3: Residual Downsampling + SE Attention ────────────────────
        mid_ch = base_ch * 4   # 256
        self.down = nn.Sequential(
            nn.Conv1d(incep_ch, mid_ch, 5, stride=2, padding=2, bias=False),
            nn.GroupNorm(min(8, mid_ch), mid_ch),
            nn.GELU(),
            ResBlock(mid_ch, k=5, dropout=dropout),
            ResBlock(mid_ch, k=5, dropout=dropout),
        )   # out: (B, 256, 250)

        self.se = SEBlock(mid_ch, reduction=16)

        # Reduce time dim: 250 → 125
        self.pool_down = nn.MaxPool1d(2)   # (B, 256, 125)

        # ── Stage 4: Transformer Encoder (4 blocks) ──────────────────────────
        self.transformer = nn.ModuleList([
            TransformerBlock(mid_ch, heads=8, dropout=dropout * 0.5)
            for _ in range(4)
        ])

        # ── Stage 5: Attention Pooling ────────────────────────────────────────
        self.attn_pool = AttentionPooling(mid_ch)   # (B, 256)

        # ── Stage 6: Metadata Fusion ──────────────────────────────────────────
        meta_embed_dim = 16
        self.meta_mlp  = MetadataMLP(meta_dim, meta_embed_dim)
        fused_dim = mid_ch + meta_embed_dim   # 272

        # ── Stage 7: Shared Embedding ─────────────────────────────────────────
        self.shared_embed = nn.Sequential(
            nn.Linear(fused_dim, mid_ch),
            nn.LayerNorm(mid_ch),
            nn.GELU(),
            nn.Dropout(dropout),
        )   # (B, 256)

        # ── Stage 8: SupCon Projection Head ───────────────────────────────────
        self.proj_head = nn.Sequential(
            nn.Linear(mid_ch, 256), nn.ReLU(),
            nn.Linear(256, 128),
        )   # (B, 128) — L2-normalised in forward

        # ── Stage 9: Multi-Task Heads ─────────────────────────────────────────
        def _head(out):
            return nn.Sequential(
                nn.Linear(mid_ch, 64), nn.GELU(), nn.Dropout(dropout * 0.5),
                nn.Linear(64, out),
            )

        self.risk_head        = _head(4)   # Low/Moderate/High/Critical
        self.arrhythmia_head  = _head(6)   # Normal SR / AF / Bradycardia / Tachycardia / PVC / Other
        self.mi_head          = _head(2)   # No MI / MI present
        self.st_t_head        = _head(3)   # Normal / ST elevation / ST depression+T-wave
        self.conduction_head  = _head(4)   # Normal / LBBB / RBBB / AVB

        # Store explainability artefacts
        self.last_attn_weights: Optional[torch.Tensor] = None   # from last transformer block
        self.last_pool_weights: Optional[torch.Tensor] = None   # from attention pooling
        self.last_se_weights:   Optional[torch.Tensor] = None   # from SE block

    def forward(self, ecg: torch.Tensor, meta: torch.Tensor) -> Dict[str, torch.Tensor]:
        # ecg:  (B, 12, 1000)
        # meta: (B, 3)

        x = self.stem(ecg)             # (B, 64, 500)
        x = self.inception(x)          # (B, 128, 500)
        x = self.down(x)               # (B, 256, 250)
        x = self.se(x)                 # (B, 256, 250) — stores se weights
        self.last_se_weights = self.se.last_weights

        x = self.pool_down(x)          # (B, 256, 125)
        x = x.transpose(1, 2)         # (B, 125, 256) — for transformer

        for blk in self.transformer:
            x = blk(x)
        # Store attention weights from last transformer block
        self.last_attn_weights = self.transformer[-1].last_attn_weights

        emb = self.attn_pool(x)        # (B, 256)
        self.last_pool_weights = self.attn_pool.last_pool_weights

        meta_emb = self.meta_mlp(meta)           # (B, 16)
        fused    = torch.cat([emb, meta_emb], 1) # (B, 272)
        shared   = self.shared_embed(fused)       # (B, 256)

        # SupCon projection (L2-normalised)
        proj = F.normalize(self.proj_head(shared), dim=1)   # (B, 128)

        return {
            "risk"       : self.risk_head(shared),        # (B, 4)
            "arrhythmia" : self.arrhythmia_head(shared),  # (B, 6)
            "mi"         : self.mi_head(shared),          # (B, 2)
            "st_t"       : self.st_t_head(shared),        # (B, 3)
            "conduction" : self.conduction_head(shared),  # (B, 4)
            "projection" : proj,                          # (B, 128)
            "embedding"  : shared,                        # (B, 256) — for t-SNE/UMAP
        }


# ── Verify forward pass ────────────────────────────────────────────────────────
model  = ECGRiskNetXAI(in_ch=12, base_ch=64, num_risk_classes=4,
                       meta_dim=3, dropout=0.3).to(device)
x_ecg  = torch.randn(4, 12, 1000).to(device)
x_meta = torch.randn(4, 3).to(device)
out    = model(x_ecg, x_meta)

print("✅ Forward pass OK")
for k, v in out.items():
    print(f"   {k:15s}: {tuple(v.shape)}")

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal parameters   : {total:,}  ({total*4/1e6:.2f} MB float32)")
print(f"Trainable parameters: {trainable:,}")
print(f"\nExplainability artefacts stored:")
print(f"  last_attn_weights : {model.last_attn_weights.shape}")
print(f"  last_pool_weights : {model.last_pool_weights.shape}")
print(f"  last_se_weights   : {model.last_se_weights.shape}")


## Loss Functions: CB-Focal Loss + SupCon Loss

In [ ]:
class ClassBalancedFocalLoss(nn.Module):
    """
    Class-Balanced Focal Loss.
    CB weighting = (1 - beta) / (1 - beta^n)  per class.
    Combined with focal modulation (1-pt)^gamma.
    """
    def __init__(self, samples_per_class, gamma=2.0, beta=0.9999, label_smoothing=0.1):
        super().__init__()
        self.gamma = gamma
        self.ls    = label_smoothing

        # CB weights
        samples_per_class = torch.tensor(samples_per_class, dtype=torch.float32)
        eff_num = 1.0 - beta ** samples_per_class
        weights = (1.0 - beta) / eff_num
        weights = weights / weights.sum() * len(weights)  # normalise
        self.register_buffer("weights", weights)

    def forward(self, logits, targets):
        C = logits.size(1)
        # Label smoothing
        with torch.no_grad():
            sm = torch.full_like(logits, self.ls / (C - 1))
            sm.scatter_(1, targets.unsqueeze(1), 1.0 - self.ls)
        log_p = F.log_softmax(logits, dim=1)
        ce    = -(sm * log_p).sum(dim=1)
        pt    = torch.exp(log_p).gather(1, targets.unsqueeze(1)).squeeze(1)
        loss  = ((1 - pt) ** self.gamma) * ce
        w     = self.weights.to(logits.device)[targets]
        return (loss * w).mean()


class SupConLoss(nn.Module):
    """
    Supervised Contrastive Loss (Khosla et al. 2020).
    Pulls same-class embeddings together, pushes different-class apart.
    """
    def __init__(self, temperature=0.07):
        super().__init__()
        self.temperature = temperature

    def forward(self, features, labels):
        # features: (B, 128) L2-normalised
        # labels:   (B,)
        device = features.device
        B = features.shape[0]

        # Similarity matrix
        sim = torch.matmul(features, features.T) / self.temperature  # (B, B)

        # Mask: same class but not self
        labels = labels.contiguous().view(-1, 1)
        pos_mask = torch.eq(labels, labels.T).float().to(device)
        pos_mask.fill_diagonal_(0)
        neg_mask = 1 - pos_mask - torch.eye(B, device=device)

        # Numerically stable logsumexp
        sim_max, _ = sim.max(dim=1, keepdim=True)
        sim        = sim - sim_max.detach()

        exp_sim  = torch.exp(sim)
        denom    = (exp_sim * (pos_mask + neg_mask)).sum(dim=1, keepdim=True)
        log_prob = sim - torch.log(denom + 1e-8)

        n_pos = pos_mask.sum(dim=1)
        loss  = -(pos_mask * log_prob).sum(dim=1) / (n_pos + 1e-8)
        return loss[n_pos > 0].mean() if (n_pos > 0).any() else torch.tensor(0.0, device=device)


# Legacy FocalLoss alias (used by NB5/NB8 which import from ecg_model.py)
class FocalLoss(nn.Module):
    """Simple Focal Loss — kept for backward compatibility."""
    def __init__(self, gamma=2.0, weight=None, label_smoothing=0.1):
        super().__init__()
        self.gamma = gamma; self.weight = weight; self.ls = label_smoothing

    def forward(self, logits, targets):
        C = logits.size(1)
        with torch.no_grad():
            sm = torch.full_like(logits, self.ls / (C - 1))
            sm.scatter_(1, targets.unsqueeze(1), 1.0 - self.ls)
        log_p = F.log_softmax(logits, dim=1)
        ce    = -(sm * log_p).sum(dim=1)
        pt    = torch.exp(log_p).gather(1, targets.unsqueeze(1)).squeeze(1)
        loss  = ((1 - pt) ** self.gamma) * ce
        if self.weight is not None:
            loss = loss * self.weight[targets]
        return loss.mean()


print("✅ CB-Focal Loss, SupCon Loss, and FocalLoss defined.")

# Quick loss test
dummy_logits = torch.randn(8, 4)
dummy_labels = torch.randint(0, 4, (8,))
dummy_projs  = F.normalize(torch.randn(8, 128), dim=1)

cbfl = ClassBalancedFocalLoss(samples_per_class=[6000, 1500, 400, 5000])
scl  = SupConLoss(temperature=0.07)

loss_cb = cbfl(dummy_logits, dummy_labels)
loss_sc = scl(dummy_projs, dummy_labels)
total_loss = loss_cb + 0.2 * loss_sc
print(f"CB-Focal loss : {loss_cb.item():.4f}")
print(f"SupCon loss   : {loss_sc.item():.4f}")
print(f"Total loss    : {total_loss.item():.4f}")


## Save Model Definition to ecg_model.py

In [ ]:
model_code = '''
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Optional, Dict


class ResBlock(nn.Module):
    def __init__(self, ch, k=7, dropout=0.2):
        super().__init__()
        p = k // 2
        self.net = nn.Sequential(
            nn.Conv1d(ch, ch, k, padding=p, bias=False),
            nn.GroupNorm(min(8, ch), ch), nn.GELU(), nn.Dropout(dropout),
            nn.Conv1d(ch, ch, k, padding=p, bias=False),
            nn.GroupNorm(min(8, ch), ch), nn.GELU(),
        )
    def forward(self, x): return x + self.net(x)


class InceptionBlock(nn.Module):
    def __init__(self, in_ch, out_ch_per_branch=32, bottleneck=32, dropout=0.1):
        super().__init__()
        self.bottleneck     = nn.Conv1d(in_ch, bottleneck, 1, bias=False)
        self.conv3          = nn.Conv1d(bottleneck, out_ch_per_branch, 3,  padding=1,  bias=False)
        self.conv7          = nn.Conv1d(bottleneck, out_ch_per_branch, 7,  padding=3,  bias=False)
        self.conv15         = nn.Conv1d(bottleneck, out_ch_per_branch, 15, padding=7,  bias=False)
        self.conv31         = nn.Conv1d(bottleneck, out_ch_per_branch, 31, padding=15, bias=False)
        self.maxpool_branch = nn.Sequential(
            nn.MaxPool1d(3, stride=1, padding=1),
            nn.Conv1d(in_ch, out_ch_per_branch, 1, bias=False),
        )
        total_out  = out_ch_per_branch * 5
        self.norm  = nn.GroupNorm(min(8, total_out), total_out)
        self.act   = nn.GELU()
        self.drop  = nn.Dropout(dropout)
        self.proj  = nn.Conv1d(total_out, out_ch_per_branch * 4, 1, bias=False)
    def forward(self, x):
        b = self.bottleneck(x)
        out = torch.cat([self.conv3(b), self.conv7(b), self.conv15(b),
                         self.conv31(b), self.maxpool_branch(x)], dim=1)
        return self.proj(self.drop(self.act(self.norm(out))))


class SEBlock(nn.Module):
    def __init__(self, ch, reduction=16):
        super().__init__()
        mid = max(ch // reduction, 4)
        self.fc = nn.Sequential(nn.Linear(ch, mid), nn.ReLU(), nn.Linear(mid, ch), nn.Sigmoid())
        self.last_weights: Optional[torch.Tensor] = None
    def forward(self, x):
        w = self.fc(x.mean(dim=-1)); self.last_weights = w.detach().cpu()
        return x * w.unsqueeze(-1)


class TransformerBlock(nn.Module):
    def __init__(self, d_model, heads=8, dropout=0.1, ffn_mult=4):
        super().__init__()
        self.attn  = nn.MultiheadAttention(d_model, heads, dropout=dropout, batch_first=True)
        self.norm1 = nn.LayerNorm(d_model); self.norm2 = nn.LayerNorm(d_model)
        self.ffn   = nn.Sequential(
            nn.Linear(d_model, d_model * ffn_mult), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_model * ffn_mult, d_model), nn.Dropout(dropout),
        )
        self.last_attn_weights: Optional[torch.Tensor] = None
    def forward(self, x):
        out, w = self.attn(x, x, x, need_weights=True, average_attn_weights=False)
        if w is not None: self.last_attn_weights = w.detach().cpu()
        x = self.norm1(x + out); x = self.norm2(x + self.ffn(x)); return x


class AttentionPooling(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.query = nn.Linear(d_model, 1)
        self.last_pool_weights: Optional[torch.Tensor] = None
    def forward(self, x):
        w = torch.softmax(self.query(x), dim=1); self.last_pool_weights = w.detach().cpu()
        return (x * w).sum(dim=1)


class MetadataMLP(nn.Module):
    def __init__(self, in_dim=3, out_dim=16):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(in_dim, 32), nn.GELU(), nn.Linear(32, out_dim), nn.GELU())
    def forward(self, m): return self.net(m)


class ECGRiskNetXAI(nn.Module):
    def __init__(self, in_ch=12, base_ch=64, num_risk_classes=4, meta_dim=3, dropout=0.3):
        super().__init__()
        self.stem      = nn.Sequential(
            nn.Conv1d(in_ch, base_ch, 15, padding=7, bias=False),
            nn.GroupNorm(min(8, base_ch), base_ch), nn.GELU(), nn.MaxPool1d(2),
        )
        self.inception = InceptionBlock(base_ch, 32, base_ch, dropout * 0.5)
        incep_ch = 128; mid_ch = base_ch * 4  # 256
        self.down      = nn.Sequential(
            nn.Conv1d(incep_ch, mid_ch, 5, stride=2, padding=2, bias=False),
            nn.GroupNorm(min(8, mid_ch), mid_ch), nn.GELU(),
            ResBlock(mid_ch, 5, dropout), ResBlock(mid_ch, 5, dropout),
        )
        self.se         = SEBlock(mid_ch, 16)
        self.pool_down  = nn.MaxPool1d(2)
        self.transformer = nn.ModuleList([TransformerBlock(mid_ch, 8, dropout * 0.5) for _ in range(4)])
        self.attn_pool   = AttentionPooling(mid_ch)
        meta_embed_dim   = 16
        self.meta_mlp    = MetadataMLP(meta_dim, meta_embed_dim)
        fused_dim        = mid_ch + meta_embed_dim
        self.shared_embed = nn.Sequential(
            nn.Linear(fused_dim, mid_ch), nn.LayerNorm(mid_ch), nn.GELU(), nn.Dropout(dropout),
        )
        self.proj_head   = nn.Sequential(nn.Linear(mid_ch, 256), nn.ReLU(), nn.Linear(256, 128))
        def _head(out): return nn.Sequential(
            nn.Linear(mid_ch, 64), nn.GELU(), nn.Dropout(dropout * 0.5), nn.Linear(64, out)
        )
        self.risk_head       = _head(4)
        self.arrhythmia_head = _head(6)
        self.mi_head         = _head(2)
        self.st_t_head       = _head(3)
        self.conduction_head = _head(4)
        self.last_attn_weights: Optional[torch.Tensor] = None
        self.last_pool_weights: Optional[torch.Tensor] = None
        self.last_se_weights:   Optional[torch.Tensor] = None

    def forward(self, ecg: torch.Tensor, meta: torch.Tensor) -> Dict[str, torch.Tensor]:
        x = self.pool_down(self.se(self.down(self.inception(self.stem(ecg)))))
        self.last_se_weights = self.se.last_weights
        x = x.transpose(1, 2)
        for blk in self.transformer: x = blk(x)
        self.last_attn_weights = self.transformer[-1].last_attn_weights
        emb  = self.attn_pool(x); self.last_pool_weights = self.attn_pool.last_pool_weights
        fused  = torch.cat([emb, self.meta_mlp(meta)], 1)
        shared = self.shared_embed(fused)
        proj   = F.normalize(self.proj_head(shared), dim=1)
        return {
            "risk": self.risk_head(shared), "arrhythmia": self.arrhythmia_head(shared),
            "mi": self.mi_head(shared), "st_t": self.st_t_head(shared),
            "conduction": self.conduction_head(shared),
            "projection": proj, "embedding": shared,
        }


class ClassBalancedFocalLoss(nn.Module):
    def __init__(self, samples_per_class, gamma=2.0, beta=0.9999, label_smoothing=0.1):
        super().__init__()
        self.gamma = gamma; self.ls = label_smoothing
        s = torch.tensor(samples_per_class, dtype=torch.float32)
        w = (1.0 - beta) / (1.0 - beta ** s)
        w = w / w.sum() * len(w)
        self.register_buffer("weights", w)
    def forward(self, logits, targets):
        C = logits.size(1)
        with torch.no_grad():
            sm = torch.full_like(logits, self.ls / (C - 1))
            sm.scatter_(1, targets.unsqueeze(1), 1.0 - self.ls)
        log_p = F.log_softmax(logits, dim=1)
        ce    = -(sm * log_p).sum(dim=1)
        pt    = torch.exp(log_p).gather(1, targets.unsqueeze(1)).squeeze(1)
        return ((loss := ((1 - pt) ** self.gamma) * ce) * self.weights.to(logits.device)[targets]).mean()


class SupConLoss(nn.Module):
    def __init__(self, temperature=0.07):
        super().__init__()
        self.temperature = temperature
    def forward(self, features, labels):
        device = features.device; B = features.shape[0]
        sim    = torch.matmul(features, features.T) / self.temperature
        labels = labels.contiguous().view(-1, 1)
        pos_mask = torch.eq(labels, labels.T).float().to(device); pos_mask.fill_diagonal_(0)
        neg_mask = 1 - pos_mask - torch.eye(B, device=device)
        sim_max, _ = sim.max(dim=1, keepdim=True); sim = sim - sim_max.detach()
        exp_sim    = torch.exp(sim)
        log_prob   = sim - torch.log((exp_sim * (pos_mask + neg_mask)).sum(dim=1, keepdim=True) + 1e-8)
        n_pos = pos_mask.sum(dim=1)
        loss  = -(pos_mask * log_prob).sum(dim=1) / (n_pos + 1e-8)
        return loss[n_pos > 0].mean() if (n_pos > 0).any() else torch.tensor(0.0, device=device)


class FocalLoss(nn.Module):
    """Legacy alias for backward compatibility."""
    def __init__(self, gamma=2.0, weight=None, label_smoothing=0.1):
        super().__init__()
        self.gamma = gamma; self.weight = weight; self.ls = label_smoothing
    def forward(self, logits, targets):
        C = logits.size(1)
        with torch.no_grad():
            sm = torch.full_like(logits, self.ls / (C - 1))
            sm.scatter_(1, targets.unsqueeze(1), 1.0 - self.ls)
        log_p = F.log_softmax(logits, dim=1)
        ce    = -(sm * log_p).sum(dim=1)
        pt    = torch.exp(log_p).gather(1, targets.unsqueeze(1)).squeeze(1)
        loss  = ((1 - pt) ** self.gamma) * ce
        if self.weight is not None: loss = loss * self.weight[targets]
        return loss.mean()
'''

with open(SAVE_DIR / "ecg_model.py", "w", encoding="utf-8") as f:
    f.write(model_code)

print("✅ ecg_model.py saved.")
print("NOTEBOOK 3 COMPLETE ✅")
print("Next → Notebook 4: Training with CB-Focal + SupCon + SWA + Temperature Scaling")
